In [1]:
import numpy as np
import os
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [2]:
TRAIN_LOCS_KEY = 'train_locs'
TRAIN_IDS_KEY = 'train_ids'
TAXON_IDS_KEY = 'taxon_ids'
TAXON_NAME_KEY = 'taxon_names'

Reading the file:

In [3]:
filepath = os.path.join(os.getcwd(), 'species_train.npz')
data = np.load(filepath, allow_pickle=True)
train_locs = data[TRAIN_LOCS_KEY]
train_ids = data[TRAIN_IDS_KEY]
taxon_ids = data[TAXON_IDS_KEY]
taxon_names = data[TAXON_NAME_KEY]

Mapping the taxon ids to taxon latin names: 

In [4]:
species_ids_names = dict(zip(data['taxon_ids'], data['taxon_names']))  # latin names of species 

Create pandas Dataframe from the data: 

In [5]:
df = pd.DataFrame({
    'latitude': train_locs[:, 0],
    'longitude': train_locs[:, 1], 
    'taxon_id': data['train_ids']
})
df['taxon_name'] = [species_ids_names[id] for id in data['train_ids'].astype(int)]
df.head()

,latitude,longitude,taxon_id,taxon_name
0,-18.286728,143.481247,31529,Lophognathus gilberti
1,-13.099798,130.783646,31529,Lophognathus gilberti
2,-13.965274,131.695145,31529,Lophognathus gilberti
3,-12.853950,132.800507,31529,Lophognathus gilberti
4,-12.196790,134.279327,31529,Lophognathus gilberti


In [6]:
df.shape

(272037, 4)

Data Cleanining: 

<small>1. Check for missing or invalid coordinates:</small>

In [7]:
df = df.dropna(subset=['latitude', 'longitude'])
df = df[(df['latitude'].between(-90, 90)) & (df['longitude'].between(-180, 180))]
df.shape

(272037, 4)

<small>2. Remove any duplicates or nearly duplicates (observations that are extremely close):</small>

In [8]:
df['lat_rounded'] = df['latitude'].round(5)
df['lon_rounded'] = df['longitude'].round(5)

In [9]:
df = df.drop_duplicates(subset=['lat_rounded', 'lon_rounded'])
df.shape

(237977, 6)

<small>4. Validate species IDs: </small>

In [10]:
df['taxon_id'].isna().sum()

np.int64(0)

<small>5. Convert to categorical labels:</small>

In [11]:
le = LabelEncoder()
df['label'] = le.fit_transform(df['taxon_id'])
df.head()

,latitude,longitude,taxon_id,taxon_name,lat_rounded,lon_rounded,label
0,-18.286728,143.481247,31529,Lophognathus gilberti,-18.28673,143.481247,278
1,-13.099798,130.783646,31529,Lophognathus gilberti,-13.09980,130.783646,278
2,-13.965274,131.695145,31529,Lophognathus gilberti,-13.96527,131.695145,278
3,-12.853950,132.800507,31529,Lophognathus gilberti,-12.85395,132.800507,278
4,-12.196790,134.279327,31529,Lophognathus gilberti,-12.19679,134.279327,278


<small>6. Only keep birds:</small>

<small>Note: Only run the next 2 blocks one time as they take a few seconds:</small>

In [12]:
taxa = pd.read_csv('taxa.csv')
birds = taxa[taxa['class'] == 'Aves']
bird_taxon_ids = set(birds['id'])
len(bird_taxon_ids)

35102

In [13]:
df = df[df['taxon_id'].isin(bird_taxon_ids)].copy()
df.shape

(151391, 7)

<small>7. Append the climate data</small>

In [14]:
df = pd.read_csv('bird_species_with_climate.csv')

<small>8. Clean the climate data</small>

In [15]:
df['Tmin_avg'] = df['Tmin_avg'].mask(df['Tmin_avg'] < -1e+30, np.nan)
df['Tmax_avg'] = df['Tmax_avg'].mask(df['Tmax_avg'] < -1e+30, np.nan)
df['Prec_avg'] = df['Prec_avg'].mask(df['Prec_avg'] < 0, np.nan)
print(f"Shape with nan data: {df.shape}")
df = df.dropna(subset=['Tmin_avg', 'Tmax_avg', 'Prec_avg'])
print(f"Shape without nan data: {df.shape}")

Shape with nan data: (151391, 10)
Shape without nan data: (150194, 10)


In [16]:
(df['Tmax_avg'] < df['Tmin_avg']).sum()

np.int64(0)

<small>9. Split the data to x and y and normalize the climate features</small>

In [17]:
X_data = df.drop(columns=['taxon_id', 'taxon_name', 'lat_rounded', 'lon_rounded', 'label'])
y_data = df['label']
climate_features = ['Tmin_avg', 'Tmax_avg', 'Prec_avg']
non_scaled_features = ['latitude', 'longitude']
# scale climate 
scaler = StandardScaler()
X_scaled = X_data.copy()
X_scaled[climate_features] = scaler.fit_transform(X_data[climate_features])
X_scaled.describe()

,latitude,longitude,Tmin_avg,Tmax_avg,Prec_avg
count,150194.000000,150194.000000,1.501940e+05,1.501940e+05,1.501940e+05
mean,15.544443,-9.343506,-1.998304e-16,-6.274977e-16,2.195107e-16
std,32.019650,95.974393,1.000003e+00,1.000003e+00,1.000003e+00
min,-75.284950,-178.060320,-6.684864e+00,-6.659755e+00,-1.640339e+00
25%,-19.905400,-96.408343,-6.481463e-01,-7.715034e-01,-6.507734e-01
50%,27.829345,-47.548143,1.810142e-02,1.161172e-01,-1.759906e-01
75%,41.539592,73.949602,6.747353e-01,7.379541e-01,3.944445e-01
max,72.515430,178.827590,2.290127e+00,2.249082e+00,1.336141e+01


<small>9. Split to train and validation sets</small>

In [18]:
X_train, X_val, y_train, y_val = train_test_split(X_scaled, y_data,
                                                  test_size=0.2,
                                                  random_state=42, 
                                                  stratify=y_data)

X_train.to_csv('X_train.csv', index=False)
X_val.to_csv('X_val.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
y_val.to_csv('y_val.csv', index=False)

Exploratory Data Analysis:

In [19]:
# Hajer

Models:

In [20]:
# Model 1:

In [21]:
# Model 2:

In [22]:
# Model 3: